# 01. Data Pull & Dataset Verification

**Football Passing Networks: Structure, Outcomes and Playing Style**

This notebook verifies the StatsBomb Open Data extraction for **FIFA World Cup 2018** and **FIFA World Cup 2022** matches.
It inspects the raw match metadata, truncated completed passing events (truncated at first substitution/red card for stable 11-player lineups), weighted directed edge lists, and team playing style metrics.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Paths
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("Checking data directories...")
print("Raw dir exists:", RAW_DIR.exists())
print("Processed dir exists:", PROCESSED_DIR.exists())

## 1. Load Processed Datasets

In [ ]:
# Load data tables
matches_df = pd.read_parquet(RAW_DIR / "matches.parquet")
team_summary_df = pd.read_parquet(PROCESSED_DIR / "team_match_summary.parquet")
edges_df = pd.read_parquet(PROCESSED_DIR / "passing_edges.parquet")
player_stats_df = pd.read_parquet(PROCESSED_DIR / "player_match_stats.parquet")
passes_df = pd.read_parquet(PROCESSED_DIR / "passing_events.parquet")

print(f"Loaded {len(matches_df)} matches across competitions.")
print(f"Loaded {len(team_summary_df)} team-match summaries.")
print(f"Loaded {len(edges_df)} passing graph edges.")
print(f"Loaded {len(player_stats_df)} player match position entries.")
print(f"Loaded {len(passes_df)} completed passing events.")

## 2. Dataset Summary & Match Outcomes

In [ ]:
print("--- Matches Breakdown by Season ---")
print(matches_df["season_name"].value_counts())

print("\n--- Team Match Results (Win / Loss / Draw) ---")
print(team_summary_df["result"].value_counts())

display(team_summary_df.head(10))

## 3. Sample Passing Edge List & Graph Construction

In [ ]:
# Select a sample team-match graph (e.g., Argentina in a match)
sample_match_id = team_summary_df.iloc[0]["match_id"]
sample_team = team_summary_df.iloc[0]["team_name"]

sample_edges = edges_df[(edges_df["match_id"] == sample_match_id) & (edges_df["team_name"] == sample_team)]
print(f"Sample Passing Edges for {sample_team} (Match ID: {sample_match_id}):")
display(sample_edges.head(10))

# Build NetworkX DiGraph
G = nx.DiGraph()
for _, row in sample_edges.iterrows():
    w = row["weight"]
    G.add_edge(row["passer_name"], row["recipient_name"], weight=w, distance=1.0 / w)

print(f"\nNetworkX Graph Built: {G.number_of_nodes()} nodes (players), {G.number_of_edges()} directed edges.")